# Text Generation LSTM Model
Next word prediction model using LSTM neural network

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import tensorflow as tf
import kagglehub
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

## 2. Download Dataset

In [ ]:
path = kagglehub.dataset_download("viictte/wikipedia-txt-dataset")
print("Dataset path:", path)

## 3. Read Text Files (LIMIT DATA)
### ✅ CHANGE 1: MAX_CHARS 2,000,000 se 200,000 kar diya
### Pehle poori 90MB file read ho rahi thi — ab sirf 200K characters

In [ ]:
all_text = ""

MAX_FILES = 5
MAX_CHARS = 200_000  # ✅ 2,000,000 → 200,000 (10x kam data)

txt_files = [f for f in os.listdir(path) if f.endswith(".txt")]
print("Total txt files found:", len(txt_files))

count_files = 0

for file_name in txt_files:
    file_path = os.path.join(path, file_name)

    with open(file_path, "r", encoding="utf-8") as f:
        data = f.read(MAX_CHARS)  # ✅ seedha MAX_CHARS tak hi read karo

    all_text += " " + data

    count_files += 1
    print(f"Loaded {count_files} file(s): {file_name}")

    if count_files >= MAX_FILES or len(all_text) >= MAX_CHARS:
        break

## 4. Basic Cleaning

In [ ]:
text = all_text.lower().replace("\n", " ")
print("Total characters loaded:", len(text))

## 5. Tokenization
### ✅ CHANGE 2: num_words=10000 add kiya — vocabulary cap
### Pehle 240,007 words the — ab sirf top 10,000 words use honge
### Isse model size 356MB se ~15MB ho jayega

In [ ]:
tokenizer = Tokenizer(num_words=10000)  # ✅ vocabulary cap: sirf top 10,000 words
tokenizer.fit_on_texts([text])

total_words = min(len(tokenizer.word_index) + 1, 10000)  # ✅ 10,000 se zyada nahi
print("Total words:", total_words)

## 6. Create Sequences

In [ ]:
input_sequences = []

sentences = text.split(".")

for line in sentences:
    if len(line.strip()) == 0:
        continue

    token_list = tokenizer.texts_to_sequences([line])[0]

    # skip very short sentences (important for stability)
    if len(token_list) < 2:
        continue

    # limit long sentences (prevents RAM spike)
    token_list = token_list[:50]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# convert to numpy
input_sequences = np.array(input_sequences, dtype=object)

# padding (single time)
max_sequence_len = max(len(x) for x in input_sequences)
input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding="pre"
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 7. Build LSTM Model

In [ ]:
model = Sequential([
    Embedding(input_dim=total_words, output_dim=128),
    LSTM(256, return_sequences=True),
    Dropout(0.2),
    LSTM(256),
    Dense(256, activation="relu"),
    Dense(total_words, activation="softmax")
])

model.build(input_shape=(None, max_sequence_len - 1))

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

## 8. Train Model

In [ ]:
model.fit(X, y, epochs=10, batch_size=128)

## 9. Save & Download
### ✅ CHANGE 3: Drive mount error fix — force_remount hata diya
### Pehle error aa raha tha: 'Mountpoint must not already contain files'

In [ ]:
model.save("text_generation_lstm.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("maxlen.pkl", "wb") as f:
    pickle.dump(max_sequence_len, f)

print("Model + tokenizer saved successfully!")
print(f"Model file size: {os.path.getsize('text_generation_lstm.h5') / 1e6:.1f} MB")

# Direct download (ab file chhoti hai to ye kaam karega)
from google.colab import files
files.download("text_generation_lstm.h5")
files.download("tokenizer.pkl")
files.download("maxlen.pkl")